In [3]:
!pip install groq pydub sounddevice scipy matplotlib pandas

In [ ]:
import os
import io
import sounddevice as sd
from scipy.io import wavfile
from pydub import AudioSegment
from groq import Groq
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Get Groq API key from environment
groq_api_key = os.getenv("GROQ_API_KEY")
if not groq_api_key:
    raise ValueError("GROQ_API_KEY environment variable is not set. Please configure it in your .env file.")

client = Groq()

# Global storage variables for the notebook runtime
notebook_transcript = ""

In [5]:
def record_audio_snippet(duration_seconds=10, sample_rate=44100):
    """Records audio from your system hardware microphone and packages it into memory."""
    print(f"🎙️ Recording active for the next {duration_seconds} seconds... Speak now!")
    
    # Capture audio arrays via sounddevice
    recording = sd.rec(int(duration_seconds * sample_rate), samplerate=sample_rate, channels=1, dtype='int16')
    sd.wait()  # Block block until the recording completes
    print("🛑 Recording finished. Packing audio data stream...")
    
    # Write the array tracking context data down into an internal byte-buffer layout
    byte_io = io.BytesIO()
    wavfile.write(byte_io, sample_rate, recording)
    return byte_io.getvalue()

# Run a sample 10-second test record right now!
audio_data_bytes = record_audio_snippet(duration_seconds=10)

🎙️ Recording active for the next 10 seconds... Speak now!
🛑 Recording finished. Packing audio data stream...


In [6]:
def process_audio_chunk(audio_bytes):
    """Sends audio bytes directly to Groq Whisper API."""
    try:
        audio_file = io.BytesIO(audio_bytes)
        audio_file.name = "audio.wav"
        translation = client.audio.transcriptions.create(
            file=audio_file,
            model="whisper-large-v3",
            response_format="text"
        )
        return translation
    except Exception as e:
        return f"Error: {e}"

def generate_structured_notes(full_transcript):
    """Extracts takeaways, actions, and questions using Llama 3.3."""
    system_prompt = """
    You are an elite Executive AI Assistant. Organize the live audio transcript text into these Markdown headers:
    ### 🎯 Key Takeaways
    ### ⚡ Action Items
    ### ❓ Follow-up Questions
    """
    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"Transcript:\n{full_transcript}"}
            ],
            temperature=0.2
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"Error: {e}"

# 1. Update text variables
new_text = process_audio_chunk(audio_data_bytes)
notebook_transcript += " " + new_text
print(f"🤖 Transcribed Text Output:\n{notebook_transcript}\n")

# 2. Extract structured analysis records
notes_summary = generate_structured_notes(notebook_transcript)
print("📚 GENERATED NOTEBOOKS DOCUMENT STRUCTURE:")
print(notes_summary)

🤖 Transcribed Text Output:
  அந்திரும் பார்க்க முடியும்

📚 GENERATED NOTEBOOKS DOCUMENT STRUCTURE:
It seems like the provided transcript is in Tamil. Here's the organized version with the requested Markdown headers:


### 🎯 Key Takeaways
The transcript provided is in Tamil and appears to be a short phrase: "அந்திரும் பார்க்க முடியும்". This translates to "You can also see" or "It's also visible" in English.


### ⚡ Action Items
There are no specific action items mentioned in the provided transcript.


### ❓ Follow-up Questions
Some potential follow-up questions could be:
* What is being referred to as "அந்திரும்" (it)?
* What is the context of the conversation?
* Is there more information or a larger conversation that this phrase is a part of?


In [8]:
import os
import io
import time
import sounddevice as sd
from scipy.io import wavfile
from groq import Groq

# Configure variables
os.environ["GROQ_API_KEY"] = "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"
client = Groq()

def interactive_recording_loop(sample_rate=44100):
    """Allows manual prompt intervention loops inside local hardware setups."""
    input("Press ENTER to 🟢 START recording...")
    print("🎙️ Recording active! Type 'stop' or press Enter again to end the session.")
    
    start_time = time.time()
    # Start a non-blocking asynchronous audio recording pool thread stream
    recording = sd.rec(int(300 * sample_rate), samplerate=sample_rate, channels=1, dtype='int16')
    
    input("Press ENTER again to 🛑 STOP recording...")
    elapsed_time = time.time() - start_time
    
    print(f"Saving {elapsed_time:.1f} seconds of captured conversation...")
    sd.stop()
    
    # Crop the long array down to the exact duration spoken
    actual_samples = int(elapsed_time * sample_rate)
    final_audio = recording[:actual_samples]
    
    byte_io = io.BytesIO()
    wavfile.write(byte_io, sample_rate, final_audio)
    return byte_io.getvalue()

# 1. Run the interactive recorder cell step
audio_data = interactive_recording_loop()

# 2. Run your inference tasks
print("\n🤖 Sending to Groq engines for analysis...")
try:
    audio_file = io.BytesIO(audio_data)
    audio_file.name = "recording.wav"
    
    transcript = client.audio.transcriptions.create(
        file=audio_file,
        model="whisper-large-v3",
        response_format="text"
    )
    print(f"\n📝 Generated Text:\n{transcript}")
except Exception as e:
    print(f"API Error: {e}")

🎙️ Recording active! Type 'stop' or press Enter again to end the session.
Saving 4.8 seconds of captured conversation...

🤖 Sending to Groq engines for analysis...

📝 Generated Text:
 Bine ați venit!
